# Évaluation & prédictions EfficientNetB0

Ce notebook charge le modèle **EfficientNetB0** entraîné pour la détection de pneumonie (normal / bacteria / virus),
reconstruit les DataLoaders via `on_the_fly_augmentation.ipynb`, puis calcule les métriques sur le **test set** et affiche quelques prédictions.

Nous utilisons les mêmes outils que pour l'entraînement :
- **PyTorch** pour la partie modèle et les DataLoaders,
- **EfficientNetB0** de `torchvision` comme architecture de backbone,
- la fonction de coût et les métriques de `scikit-learn` pour analyser la performance du modèle.

L'objectif est de calculer plusieurs métriques de classification (Accuracy, Precision, Recall, F1-score, ROC-AUC) et de les visualiser sous forme de **graphique en barres**.

In [ ]:
# Recharge le pipeline d’augmentation et les DataLoaders
# Cette commande exécute le notebook d'augmentation pour recréer `test_loader`
%run "../on_the_fly_augmentation.ipynb"

## 1. Reconstruction du modèle et chargement des poids

On reconstruit exactement la même architecture **EfficientNetB0** que dans le notebook d'entraînement :
- même nombre de classes (3),
- même remplacement de la couche de classification finale,
- même initialisation par un backbone pré‑entraîné sur ImageNet.

Ensuite, on charge les poids appris (`efficientnet_b0_pneumonia.pt`) et on met le modèle en mode `eval()` pour figer les poids pendant l'évaluation.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

NUM_CLASSES = 3
class_names = ["normal", "bacteria", "virus"]

# Reconstruit le modèle et recharge les poids
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
model = efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

model_path = "./efficientnet_b0_pneumonia.pt"
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f"Poids chargés depuis {model_path}")

In [ ]:
# Évaluation sur le test set
# On calcule ici : Accuracy, Precision, Recall, F1-score et ROC-AUC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

all_labels = []
all_preds = []
all_probs = []  # probabilités (softmax) nécessaires pour le ROC-AUC

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# Métriques globales (macro pour les métriques par classe)
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

# ROC-AUC multi-classe (one-vs-rest)
try:
    roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    roc_auc = float("nan")

print("\nClassification report (par classe) :")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\nGlobal metrics (macro-average) :")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - test set")
plt.tight_layout()
plt.show()

# Graphique en barres des métriques (en pourcentage)
metrics_names = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
metrics_values = [accuracy, precision, recall, f1, roc_auc]

# Conversion en pourcentage
metrics_percent = [m * 100 for m in metrics_values]

plt.figure(figsize=(8, 5))
plt.bar(metrics_names, metrics_percent, color=["#4caf50", "#2196f3", "#ff9800", "#9c27b0", "#f44336"])
plt.ylabel("Score (%)")
plt.ylim(0, 100)
plt.title("Global evaluation metrics (test set)")
for i, v in enumerate(metrics_percent):
    plt.text(i, v + 1, f"{v:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 2. Visualisation qualitative de quelques prédictions

Pour compléter les métriques globales, on affiche quelques images du test set avec :
- le **label réel**,
- la **prédiction du modèle**.

Cela permet de vérifier visuellement le comportement du modèle sur des exemples concrets (y compris les éventuelles erreurs).

In [ ]:
# Visualisation de quelques prédictions

num_examples = 6
shown = 0

plt.figure(figsize=(12, 6))

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        for i in range(images.size(0)):
            if shown >= num_examples:
                break

            img = images[i].cpu().numpy().transpose(1, 2, 0)
            # On inverse la normalisation ImageNet grossièrement
            img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img = np.clip(img, 0, 1)

            true_label = class_names[labels[i].item()]
            pred_label = class_names[preds[i].item()]

            plt.subplot(2, 3, shown + 1)
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"Vrai: {true_label}\nPrédit: {pred_label}")

            shown += 1
        if shown >= num_examples:
            break

plt.tight_layout()
plt.show()